# Sélectionner et filtrer

Notebook court. L'objectif est de poser **l'idiome moderne** utilisé dans tout le reste du module,
pas de faire le tour complet de l'API.

Pour les subtilités (`.iloc`, `.where()`, masques combinés avancés) : dossier `advance/`.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from _data import load_accidents

df = load_accidents()
df.shape

## 1. Sélectionner des colonnes

In [ ]:
# Une colonne → Series
df["dep"]

In [ ]:
# Plusieurs colonnes → DataFrame (double crochet)
df[["dep", "mois", "lum"]].head()

In [ ]:
# Sélection dynamique par nom de colonne : .filter(like="substring")
# Utile quand les colonnes suivent une convention de nommage
# Ici : toutes les colonnes dont le nom contient "pr" (pr, pr1, prof)
df.filter(like="pr").head(3)

In [ ]:
# Pour un préfixe ou suffixe strict, utiliser .filter(regex=)
# Exemple : uniquement les colonnes qui commencent par "pr"
df.filter(regex="^pr").head(3)

`.filter()` est particulièrement utile en fin de pipeline quand les colonnes résultat suivent une convention —
par exemple après un `.agg()` qui produit `nb_accidents`, `nb_voie_degradee`, `pct_voie_degradee` :
on peut écrire `.filter(like="nb_")` pour ne garder que les colonnes de comptage.

## 2. Filtrer des lignes avec `.query()`

`.query()` est **la méthode par défaut** dans ce cours — pas une alternative.
Elle est lisible, chaînable, et proche du SQL.

In [ ]:
# Condition simple
df.query("dep == '75'").shape  # accidents à Paris

In [ ]:
# Conditions combinées
df.query("dep == '75' and lum == 5").shape  # Paris, nuit sans éclairage

In [ ]:
# Plage de valeurs
df.query("mois in [6, 7, 8]").shape  # accidents estivaux

In [ ]:
# Variable Python dans la condition : préfixe @
dep_cible = "69"  # Rhône
df.query("dep == @dep_cible").shape

### Gotcha : nom de colonne qui contient un espace

`.query()` ne sait pas parser `"ma colonne == 1"` si le nom contient un espace.
La solution : encadrer le nom avec des backticks.

In [ ]:
# Exemple avec un DataFrame jouet — nos vraies données n'ont pas ce problème,
# mais vous le croiserez en entreprise sur des exports Excel.
df_exemple = pd.DataFrame({"code dep": [75, 69, 13], "nb": [10, 5, 8]})

df_exemple.query("`code dep` == 75")

## 3. Filtrer avec un masque booléen

L'autre syntaxe existe — un LLM l'utilise parfois, il faut savoir la lire.

In [ ]:
# Masque booléen — même résultat que .query("dep == '75'")
df[df["dep"] == "75"].shape

In [ ]:
# Conditions combinées avec masques : & (et), | (ou), ~ (non)
# Chaque condition doit être entre parenthèses
df[(df["dep"] == "75") & (df["lum"] == 5)].shape

**Règle pratique** : utilisez `.query()` par défaut.
Utilisez un masque booléen quand `.query()` ne sait pas l'exprimer —
par exemple avec `.str.contains()`, `.isin()` sur une liste calculée dynamiquement,
ou une condition qui appelle une fonction Python.

In [ ]:
# Exemple typique où le masque est plus naturel : .str.contains()
df[df["adr"].str.contains("République", na=False)].head(3)

## 4. Sélectionner par label : `.loc[]`

`.loc[filtre_lignes, filtre_colonnes]` permet de filtrer des lignes **et** sélectionner des colonnes
en une seule opération.

In [ ]:
# Filtrer les lignes ET sélectionner des colonnes en une passe
df.loc[df["dep"] == "75", ["Num_Acc", "mois", "lum", "col"]].head()

En pratique, l'équivalent chaîné est souvent plus lisible :

```python
df.query("dep == '75'")[["Num_Acc", "mois", "lum", "col"]]
```

`.loc` et `.iloc` ont des subtilités (sélection par position entière, slicing d'index, etc.)
couvertes dans `advance/3.1-selection_subtilites_loc_iloc.ipynb` si vous voulez creuser.
En pratique, `.query()` + sélection de colonnes couvre 90 % des cas.

## Quiz

<details><summary>Question 1 — Quelle est la différence entre <code>df["dep"]</code> et <code>df[["dep"]]</code> ?</summary>

`df["dep"]` retourne une **Series** (une seule colonne, une dimension).
`df[["dep"]]` retourne un **DataFrame** (toujours deux dimensions, même avec une seule colonne).
La distinction compte dès qu'une fonction attend l'un ou l'autre type.

</details>

<details><summary>Question 2 — Quand préférer un masque booléen à <code>.query()</code> ?</summary>

Quand la condition ne peut pas s'écrire comme une chaîne de caractères simple :
`.str.contains()`, `.isin()` avec une liste calculée à la volée, appel de fonction Python arbitraire.
Dans tous les autres cas, `.query()` est plus lisible et chaînable.

</details>